In [6]:
import json
import math
import pandas as pd
import numpy as np
from shapely.geometry import shape, Point
import gamspy as gp

# ----------------------------------------------------
# 1. Load Data & Geometries
# ----------------------------------------------------
# Load district population
df_pop = pd.read_excel(f"Data\جمعیت مناطق-1405-06-25.xlsx")
pop_dict = {
    1: 80807, 2: 76733, 3: 111013, 4: 147139, 5: 132984,
    6: 112896, 7: 206588, 8: 243563, 9: 78271, 10: 200702,
    11: 58334, 12: 155413, 13: 164413, 14: 160354, 15: 138247
}

# Parse 15 district polygons and calculate centroids
with open(r"Data\isfahan_raw.geojson", "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

persian_to_eng = {
    '۱': 1, '۲': 2, '۳': 3, '۴': 4, '۵': 5, '۶': 6, '۷': 7, '۸': 8,
    '۹': 9, '۱۰': 10, '۱۱': 11, '۱۲': 12, '۱۳': 13, '۱۴': 14, '۱۵': 15
}

districts = []
for feat in geojson_data["features"]:
    geom_type = feat.get("geometry", {}).get("type")
    props = feat.get("properties", {})
    name = props.get("name", "")
    admin_level = str(props.get("admin_level", ""))

    if geom_type not in ["Polygon", "MultiPolygon"] or admin_level != "9" or "خمینی" in name:
        continue

    d_num = None
    for p_num, val in persian_to_eng.items():
        if f"منطقه {p_num}" == name.strip() or f"منطقه {p_num} " in name:
            d_num = val
            break
    if d_num:
        poly = shape(feat["geometry"])
        districts.append({
            "id": f"D{d_num}",
            "d_num": d_num,
            "name": name,
            "lat": poly.centroid.y,
            "lon": poly.centroid.x,
            "geom": poly,
            "population": pop_dict[d_num]
        })

df_districts = pd.DataFrame(districts).sort_values("d_num").reset_index(drop=True)

In [8]:
refined_targets = np.load(r"Data\refined_targets.npy", allow_pickle=True)
target_counts = {d["id"]: 0 for d in districts}
for t in refined_targets:
    pt = Point(t["lon"], t["lat"])
    for d in districts:
        if d["geom"].contains(pt):
            target_counts[d["id"]] += 1
            break
refined_targets

array([{'id': 297227347, 'name': 'Badr Air Base', 'category': 'Aviation & Air Bases', 'type': 'airfield', 'lat': 32.6209518, 'lon': 51.6875522},
       {'id': 5643934331, 'name': 'Military/Defense Facility', 'category': 'Defense & Military', 'type': 'bunker', 'lat': 32.5659344, 'lon': 51.6541672},
       {'id': 6380568145, 'name': 'Military/Defense Facility', 'category': 'Defense & Military', 'type': 'checkpoint', 'lat': 32.5987561, 'lon': 51.6640166},
       {'id': 8495731262, 'name': 'بسیج', 'category': 'Defense & Military', 'type': 'complex', 'lat': 32.5529796, 'lon': 51.5019455},
       {'id': 8515927623, 'name': 'بسیج', 'category': 'Defense & Military', 'type': 'complex', 'lat': 32.6653343, 'lon': 51.5817841},
       {'id': 8515927624, 'name': 'بسیج', 'category': 'Defense & Military', 'type': 'complex', 'lat': 32.6653326, 'lon': 51.5817036},
       {'id': 8722031272, 'name': 'پایگاه هوایی هشتم شکاری', 'category': 'Aviation & Air Bases', 'type': 'airfield', 'lat': 32.7232001, 'lon'

In [3]:
x = Variable(
    container=m,
    name="x",
    domain=[i, j],
    type="Positive",
    description="amount of commodity to ship from plant i to market j",
)

In [4]:
supply = Equation(
    container=m, name="supply", domain=i, description="observe supply limit at plant i"
)
demand = Equation(
    container=m, name="demand", domain=j, description="satisfy demand at market j"
)

In [6]:
supply[i] = Sum(j, x[i, j]) <= a[i]
demand[j] = Sum(i, x[i, j]) >= b[j]

In [7]:
print(demand.latexRepr())

$
\sum_{i} x_{i,j} \geq b_{j}\hfill \forall j
$


In [8]:
obj = Sum((i, j), c[i, j] * x[i, j])

In [9]:
transport = Model(
    m,
    name="transport",
    equations=[supply, demand],
    problem="LP",
    sense=Sense.MIN,
    objective=obj,
)

In [10]:
transport = Model(
    m,
    name="transport",
    equations=m.getEquations(),
    problem="LP",
    sense=Sense.MIN,
    objective=obj,
)

In [11]:
i.setRecords(['seattle', 'san-diego'])
j.setRecords(['new-york', 'chicago', 'topeka'])
a.setRecords([("seattle", 350), ("san-diego", 600)])
b.setRecords([("new-york", 325), ("chicago", 300), ("topeka", 275)])
i.records

,uni,element_text
0,seattle,
1,san-diego,


In [12]:
import pandas as pd

distances = pd.DataFrame(
    [
        ["seattle", "new-york", 2.5],
        ["seattle", "chicago", 1.7],
        ["seattle", "topeka", 1.8],
        ["san-diego", "new-york", 2.5],
        ["san-diego", "chicago", 1.8],
        ["san-diego", "topeka", 1.4],
    ],
    columns=["from", "to", "distance"]
).set_index(["from", "to"])
d = Parameter(
    container=m,
    name="d",
    domain=[i, j],
    description="distance between plant i and market j",
    records=distances.reset_index(),
)
d.records

,from,to,value
0,seattle,new-york,2.5
1,seattle,chicago,1.7
2,seattle,topeka,1.8
3,san-diego,new-york,2.5
4,san-diego,chicago,1.8
5,san-diego,topeka,1.4


In [13]:
freight_cost = 90
cost = freight_cost * distances / 1000
c.setRecords(cost.reset_index())
c.records

,from,to,value
0,seattle,new-york,0.225
1,seattle,chicago,0.153
2,seattle,topeka,0.162
3,san-diego,new-york,0.225
4,san-diego,chicago,0.162
5,san-diego,topeka,0.126


In [14]:
c[i, j] = freight_cost * d[i, j] / 1000
c.records

,i,j,value
0,seattle,new-york,0.225
1,seattle,chicago,0.153
2,seattle,topeka,0.162
3,san-diego,new-york,0.225
4,san-diego,chicago,0.162
5,san-diego,topeka,0.126


In [15]:
import sys

transport.solve(output=sys.stdout)

--- Job _VPwtUnkdSV_pjGlT7iC_Kw.gms Start 09/16/26 19:10:48 54.4.0 9a60ae27 WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\Users\shaya\Desktop\GAMS\.gamspy_venv\Lib\site-packages\gamspy_base\gmsprmNT.txt
--- GAMS Parameters defined
    LP cplex
    Input C:\Users\shaya\AppData\Local\Temp\tmpcbdqqui6\_VPwtUnkdSV_pjGlT7iC_Kw.gms
    Output C:\Users\shaya\AppData\Local\Temp\tmpcbdqqui6\_VPwtUnkdSV_pjGlT7iC_Kw.lst
    ScrDir C:\Users\shaya\AppData\Local\Temp\tmpcbdqqui6\tmpyo8m3mvc\
    SysDir C:\Users\shaya\Desktop\GAMS\.gamspy_venv\Lib\site-packages\gamspy_base\
    LogOption 3
    License C:\Users\shaya\Desktop\GAMS\.gamspy_venv\Lib\site-packages\gamspy_base\gamslice.txt
    OptFile 0
    OptDir C:\Users\shaya\AppData\Local\Temp\tmpcbdqqui6\
    LimRow 0
    LimCol 0
    GDX C:\Users\shaya\AppData\Local\Temp\tmpcbdqqui6\_VPwtUnkdSV_pjGlT7iC_Kwout.gdx
    SolPrint 0
    SolveLink 2
    PreviousWork 1
    gdxSymbols newOrChangedNoData
System information: 4 physical cores and 8 Gb physi

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,NormalCompletion,OptimalGlobal,153.675,6.0,7.0,LP,CPLEX,0.538
